# CF-conventions `xarray` datasets from the `waterdata` module

The `dataretrieval.waterdata.xarray` module mirrors the `waterdata` getters
(`get_daily`, `get_continuous`, `get_peaks`, `get_samples`, …) but returns a
[CF-conventions](https://cfconventions.org/) [`xarray.Dataset`](https://docs.xarray.dev/)
instead of a `pandas.DataFrame`. Units, statistics, and station coordinates come
straight from the data, and parameter names from a small cached lookup; all are
written onto the dataset as CF attributes, so the result is self-describing and
ready to write to netCDF.

By **default** the data is returned on a `(monitoring_location_id, time)` grid —
one named variable per parameter, so `ds["discharge"].sel(time=...)` just works.
For large, very ragged multi-site pulls, pass `dense=False` to get the compact CF
*contiguous ragged array* instead.

This notebook covers:

1. a single time series (the default dense grid);
2. multiple parameters and sites on one grid;
3. the **ragged** layout (`dense=False`) for large, very ragged pulls;
4. selecting one series from a ragged dataset;
5. water-quality samples (`get_samples`);
6. writing CF netCDF.

## Setup

The xarray helpers are an optional extra:

```bash
pip install dataretrieval[xarray]
```

As with the rest of `waterdata`, signing up for a free
[API key](https://api.waterdata.usgs.gov/signup/) gives you higher rate limits.
Import the xarray wrappers under a short alias so they don't shadow the plain
`DataFrame`-returning getters:

In [ ]:
from dataretrieval.waterdata import xarray as wdx

## 1. A single time series

Pull a year of daily mean discharge (parameter `00060`, statistic `00003`) at one
gage. The wrapper takes the same arguments as `waterdata.get_daily` and returns a
CF dataset:

In [ ]:
ds = wdx.get_daily(
    monitoring_location_id="USGS-05407000",
    parameter_code="00060",  # discharge
    statistic_id="00003",  # daily mean
    time="2023-01-01/2024-01-01",
)
ds

The result is a CF `(monitoring_location_id, time)` grid
(`featureType = "timeSeries"`):

* each parameter is its own **named variable** (`discharge`), so `time` is a real
  dimension you can index by label;
* the descriptors derived from the data land on the variable as CF attributes
  (`long_name`, `units`, `cell_methods`, `standard_name`, `usgs_parameter_code`);
* `monitoring_location_id` is the instance dimension and carries
  `cf_role = "timeseries_id"`; the flag columns (`qualifier`, `approval_status`)
  are linked as `ancillary_variables`;
* dataset attributes carry provenance (`Conventions`, `references`,
  `date_modified`).

In [ ]:
print(ds["discharge"].attrs)
print({k: ds.attrs[k] for k in ("Conventions", "featureType", "references")})

# time is a real dimension -> label-based selection just works
ds["discharge"].sel(time="2023-07-01")

## 2. Multiple parameters and sites

Ask for several parameters across several gages in one call. Each parameter
becomes its own named variable on a shared `(monitoring_location_id, time)`
grid, NaN where a series has no observation (e.g. a gage that does not report a
given parameter):

In [ ]:
grid = wdx.get_daily(
    monitoring_location_id=["USGS-05407000", "USGS-03339000"],
    parameter_code=["00060", "00010"],  # discharge + water temperature
    statistic_id="00003",
    time="2023-01-01/2024-01-01",
)
print("dims      :", dict(grid.sizes))
print("variables :", list(grid.data_vars))
grid

Now any slice is a labelled `.sel(...)` — one site's series, or all sites on a
given day as a vector:

In [ ]:
# one site's discharge series
q = grid["discharge"].sel(monitoring_location_id="USGS-05407000")
print("single series :", dict(q.sizes))

# every site's discharge on one day
grid["discharge"].sel(time="2023-07-01")

## 3. The ragged layout for large, very ragged pulls

The dense grid is ergonomic, but it pays for a union time axis and NaN fill.
Real collections are *ragged*: some gages have a century of record, others a few
years. For large multi-site pulls, pass `dense=False` to get the CF **contiguous
ragged array**, which stores only real observations — no NaN fill.

Pull discharge at two gages with very different start dates:

In [ ]:
sites = ["USGS-05407000", "USGS-02238500"]  # records since 1913 vs 2008
# Bound the window so the docs build stays light; the start-date gap still
# shows up as different row_size.
ragged = wdx.get_daily(
    monitoring_location_id=sites,
    parameter_code="00060",
    statistic_id="00003",
    time="2000-01-01/2024-01-01",
    dense=False,
)
print("dims     :", dict(ragged.sizes))
print("row_size :", dict(zip(
    ragged["monitoring_location_id"].values, ragged["row_size"].values)))
ragged

In the ragged array every observation lives along a single `obs` dimension; each
series — one `(monitoring location, parameter, statistic)` — is one instance
along `timeseries`, and `row_size` records how many observations it contributes
(the CF `sample_dimension` link). Because it stores no NaN fill, it stays compact
as collections get ragged. Compare the in-memory size against the dense grid:

In [ ]:
dense = wdx.get_daily(
    monitoring_location_id=sites,
    parameter_code="00060",
    statistic_id="00003",
    time="2000-01-01/2024-01-01",
)  # default grid
print(f"ragged {ragged.nbytes/1e6:6.2f} MB   dense {dense.nbytes/1e6:6.2f} MB")

With two gages the gap is modest, but it grows fast: a whole state's daily
discharge spans gages with wildly different record lengths, where the gridded
layout balloons (mostly NaN) while the ragged array stores only real
observations. **Rule of thumb:** the default grid for ergonomic, label-based
slicing of a few series; `dense=False` for storage and large, very ragged
multi-site pulls.

## 4. Selecting one series from a ragged dataset

In the ragged layout `time` is a flat `obs` axis, not a dimension, so you can't
`.sel(time=...)` directly. Use `wdx.select_series` to pull one series by its
labels; it returns a tidy, time-indexed dataset on which `.sel(time=...)` works:

In [ ]:
one = wdx.select_series(
    ragged, monitoring_location_id="USGS-05407000", parameter_code="00060"
)
print("dims :", dict(one.sizes))
one["value"].sel(time="2020-07").mean()

For richer work, the [`cf-xarray`](https://cf-xarray.readthedocs.io/) package
understands the CF ragged encoding (`cf_role`, `sample_dimension`) and will
regroup/decode the whole dataset for you.

### The whole collection at once: `to_awkward`

For analysis across *all* series at once, convert the ragged dataset to an
[awkward](https://awkward-array.org/) array. The contiguous-ragged layout
(`row_size` offsets + a flat `obs` axis) *is* awkward's jagged layout, so this is
a near-zero-copy re-view: each series becomes one record — its identity as scalar
fields plus an `obs` list of `{time, value, …}` observations — so per-series
operations vectorize with no NaN fill. `awkward` is an optional dependency that
is *not* installed with `dataretrieval` (`pip install awkward`):

```python
import awkward as ak

arr = wdx.to_awkward(ragged)            # one record per series
arr[0].parameter_code                   # a series' metadata
ak.mean(arr.obs.value, axis=1)          # per-series means, all at once
arr[arr.parameter_code == "00060"]      # filter series by metadata
```

## 5. Water-quality samples

`get_samples` returns discrete water-quality results. Samples are irregular in
time and characteristic, so this getter is always ragged: one instance per
`(monitoring location, characteristic, sample fraction)`, with the result value
plus `detection_condition` and `status` as ancillary (censoring) variables.
Characteristics are free text, so no CF `standard_name` is inferred and
non-numeric results coerce to NaN (the `detection_condition` variable preserves
non-detects).

In [ ]:
wq = wdx.get_samples(
    service="results",
    profile="basicphyschem",
    monitoringLocationIdentifier="USGS-05406500",
    activityStartDateLower="2019-01-01",
    activityStartDateUpper="2020-01-01",
)
print("dims:", dict(wq.sizes))
print("characteristics:", sorted(set(wq["characteristic"].values))[:8], "...")
print("ancillary:", wq["value"].attrs.get("ancillary_variables"))

## 6. Writing CF netCDF

Because the dataset already carries CF metadata, it serializes straight to a
self-describing netCDF file that CF-aware tools (THREDDS, `cf-xarray`, Panoply,
…) can read back. This works for either layout (requires a netCDF backend, e.g.
`pip install netCDF4`):

In [ ]:
import os
import tempfile

path = os.path.join(tempfile.mkdtemp(), "daily_discharge.nc")
ds.to_netcdf(path)  # the dense grid from section 1
print("wrote", path)

## References

* CF conventions — [Discrete Sampling Geometries](https://cfconventions.org/cf-conventions/cf-conventions.html#discrete-sampling-geometries)
  (contiguous ragged array representation)
* [`cf-xarray`](https://cf-xarray.readthedocs.io/) — decode/group CF DSG datasets
* [`xarray`](https://docs.xarray.dev/) documentation